In [1]:
#lowercase
#punctuation removal
#tokenization
#stopwords
#stemming / lemmatization
#TF-IDF

In [2]:
import pandas as pd
import string

import nltk
from nltk.corpus import stopwords

from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

In [3]:
df = pd.read_csv("../data/sms-spam-collection.csv")

In [4]:
df_no_drop = df.drop_duplicates()

Lowercase, Punctuation Removal

In [5]:
def remove_punctuation(message):

    message = message.lower()
    result= []

    for char in message:
        if char not in string.punctuation:
            result.append(char)

    cleaned_txt = ''.join(result)
    return cleaned_txt


In [6]:
df['message_clean'] = df['message'].apply(remove_punctuation)

In [7]:
df[['message', 'message_clean']].head()

,message,message_clean
0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...


Tokenization, Stopword Removal

In [8]:
df['message_tokens'] = df['message_clean'].apply(lambda x: x.split())

In [9]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Ghoghnous
[nltk_data]     System\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
stop_words = set(stopwords.words('english'))


def remove_stopword(tokens):
    words = []

    for word in tokens:
        if word not in stop_words:
            words.append(word)

    return  words

In [11]:
df['message_no_stopwords'] = df['message_tokens'].apply(remove_stopword)

In [12]:
df[['message_tokens', 'message_no_stopwords']].head(10)

,message_tokens,message_no_stopwords
0,"[go, until, jurong, point, crazy, available, o...","[go, jurong, point, crazy, available, bugis, n..."
1,"[ok, lar, joking, wif, u, oni]","[ok, lar, joking, wif, u, oni]"
2,"[free, entry, in, 2, a, wkly, comp, to, win, f...","[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,"[u, dun, say, so, early, hor, u, c, already, t...","[u, dun, say, early, hor, u, c, already, say]"
4,"[nah, i, dont, think, he, goes, to, usf, he, l...","[nah, dont, think, goes, usf, lives, around, t..."
5,"[freemsg, hey, there, darling, its, been, 3, w...","[freemsg, hey, darling, 3, weeks, word, back, ..."
6,"[even, my, brother, is, not, like, to, speak, ...","[even, brother, like, speak, treat, like, aids..."
7,"[as, per, your, request, melle, melle, oru, mi...","[per, request, melle, melle, oru, minnaminungi..."
8,"[winner, as, a, valued, network, customer, you...","[winner, valued, network, customer, selected, ..."
9,"[had, your, mobile, 11, months, or, more, u, r...","[mobile, 11, months, u, r, entitled, update, l..."


Lemmatization

In [13]:
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to C:\Users\Ghoghnous
[nltk_data]     System\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Ghoghnous
[nltk_data]     System\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [14]:
lemmatizer = WordNetLemmatizer()

def lemmatized_message(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

In [15]:
df['message_lemmatized'] = df['message_no_stopwords'].apply(lemmatized_message)

In [16]:
df[['message_lemmatized', 'message_no_stopwords']].head(20)

,message_lemmatized,message_no_stopwords
0,"[go, jurong, point, crazy, available, bugis, n...","[go, jurong, point, crazy, available, bugis, n..."
1,"[ok, lar, joking, wif, u, oni]","[ok, lar, joking, wif, u, oni]"
2,"[free, entry, 2, wkly, comp, win, fa, cup, fin...","[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,"[u, dun, say, early, hor, u, c, already, say]","[u, dun, say, early, hor, u, c, already, say]"
4,"[nah, dont, think, go, usf, life, around, though]","[nah, dont, think, goes, usf, lives, around, t..."
5,"[freemsg, hey, darling, 3, week, word, back, i...","[freemsg, hey, darling, 3, weeks, word, back, ..."
6,"[even, brother, like, speak, treat, like, aid,...","[even, brother, like, speak, treat, like, aids..."
7,"[per, request, melle, melle, oru, minnaminungi...","[per, request, melle, melle, oru, minnaminungi..."
8,"[winner, valued, network, customer, selected, ...","[winner, valued, network, customer, selected, ..."
9,"[mobile, 11, month, u, r, entitled, update, la...","[mobile, 11, months, u, r, entitled, update, l..."


In [17]:
df['message_final'] = df['message_lemmatized'].apply(
    lambda words: ' '.join(words)
    )

In [18]:
df[['message_final','message_lemmatized']].head()

,message_final,message_lemmatized
0,go jurong point crazy available bugis n great ...,"[go, jurong, point, crazy, available, bugis, n..."
1,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]"
2,free entry 2 wkly comp win fa cup final tkts 2...,"[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,u dun say early hor u c already say,"[u, dun, say, early, hor, u, c, already, say]"
4,nah dont think go usf life around though,"[nah, dont, think, go, usf, life, around, though]"


Bag Of Words

In [19]:
cv = CountVectorizer()

In [ ]:
X = cv.fit_transform(df['message_final'])

In [34]:
df.to_csv('../data/processed_sms.csv', index=False, na_rep='')

In [22]:
X.shape

(5572, 8910)

In [23]:
cv.get_feature_names_out()[1000:1100]

array(['activity', 'actor', 'actual', 'actually', 'acwicmb3cktz8r74',
       'ad', 'adam', 'add', 'addamsfa', 'added', 'addicted', 'addie',
       'adding', 'address', 'addressull', 'adewale', 'adi', 'adjustable',
       'admin', 'administrator', 'admirer', 'admission', 'admit',
       'admiti', 'adore', 'adoring', 'adp', 'adress', 'adrian', 'adrink',
       'adsense', 'adult', 'advance', 'adventure', 'adventuring',
       'advice', 'advise', 'advising', 'advisor', 'aeronautics',
       'aeroplane', 'afew', 'affair', 'affection', 'affectionate',
       'affectionsamp', 'affidavit', 'afford', 'afghanistan', 'afraid',
       'africa', 'african', 'aft', 'afternon', 'afternoon', 'afterwards',
       'aftr', 'ag', 'againcall', 'againloving', 'agalla', 'age', 'age16',
       'age16150ppermesssubscription', 'age23', 'agency', 'agent',
       'agesring', 'agidhane', 'aging', 'ago', 'agocusoon', 'agree',
       'agreen', 'ah', 'aha', 'ahead', 'ahgee', 'ahhh', 'ahhhhjust',
       'ahmad', 'ahnow